# Class 2 practical lab — HTTP, FastAPI and SQLite

**Core lab time:** 40 minutes · **Course project:** Engineering Knowledge Assistant

**Environment:** Run this notebook with the Class 2 project's `.venv` kernel after installing the Class 2 dependencies listed in `README.md`.

**By the end:** inspect HTTP request/response behavior, define validated API data, persist records in SQLite, expose them through FastAPI, and test the API without starting a real network server.

> **Teaching example:** this notebook builds a tiny *notes API*. It is intentionally separate from the Engineering Knowledge Assistant homework, so the worked code does not implement the document catalogue, ingestion pipeline or search assignment.


## Check the environment

The notebook needs FastAPI, Pydantic and HTTPX (used by FastAPI's test client). If an import fails, select the `.venv` kernel from the Class 2 starter project and follow the dependency setup in `class_2/README.md`.


In [ ]:
import platform
import sqlite3
import sys
import tempfile
from pathlib import Path

import fastapi
import httpx
import pydantic

print(f"Python  : {platform.python_version()}")
print(f"FastAPI : {fastapi.__version__}")
print(f"Pydantic: {pydantic.__version__}")
print(f"HTTPX   : {httpx.__version__}")
print(f"Kernel  : {sys.executable}")


## 1. HTTP: request in, response out

An HTTP request has a method, path, headers and (sometimes) a body. A response has a status code, headers and (often) a body.

| Method | Typical intent | Example |
|---|---|---|
| `GET` | Read a resource | `GET /notes/12` |
| `POST` | Create or trigger work | `POST /notes` |
| `PUT` | Replace a resource | `PUT /notes/12` |
| `PATCH` | Partially update | `PATCH /notes/12` |
| `DELETE` | Remove a resource | `DELETE /notes/12` |

Common status codes: `200` OK, `201` Created, `204` No Content, `400` Bad Request, `404` Not Found, `422` Validation Error and `500` Internal Server Error.

The following object is only a small representation of a request; it is not a real network call.


In [ ]:
request_example = {
    "method": "POST",
    "path": "/notes",
    "headers": {"content-type": "application/json"},
    "body": {"title": "API contract", "body": "A request creates a note."},
}
response_example = {
    "status_code": 201,
    "headers": {"content-type": "application/json"},
    "body": {"id": 1, **request_example["body"]},
}
print("REQUEST:", request_example)
print("RESPONSE:", response_example)


## 2. Pydantic: validate the shape of data

An API should not trust incoming JSON. Pydantic models describe the expected shape and validate values before your route handles them.

We will use notes instead of documents in this guided example.


In [ ]:
from pydantic import BaseModel, Field, ValidationError


class NoteCreate(BaseModel):
    title: str = Field(min_length=1, max_length=80)
    body: str = Field(min_length=1)


valid_note = NoteCreate(title="HTTP basics", body="A response has a status code.")
print(valid_note.model_dump())

try:
    NoteCreate(title="", body="A title is required.")
except ValidationError as error:
    print("Validation rejected the payload:")
    print(error)


## 3. SQLite: persist data rather than keeping it in memory

SQLite stores relational data in a local file. For this lab, create a temporary database so repeated notebook runs do not leave project data behind.

A parameterized query uses placeholders (`?`) for values. Do not build SQL by concatenating user input.


In [ ]:
temporary_directory = tempfile.TemporaryDirectory(prefix="class2-notes-lab-")
DB_PATH = Path(temporary_directory.name) / "notes.sqlite3"


def connect_database() -> sqlite3.Connection:
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    return connection


with connect_database() as connection:
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            body TEXT NOT NULL
        )
        """
    )

print("Database created:", DB_PATH.exists())


### A small repository layer

The repository owns SQL. The API route should call these functions rather than embedding SQL inside the route handler.


In [ ]:
def create_note(title: str, body: str) -> int:
    with connect_database() as connection:
        cursor = connection.execute(
            "INSERT INTO notes (title, body) VALUES (?, ?)",
            (title, body),
        )
        return int(cursor.lastrowid)


def list_notes() -> list[dict[str, object]]:
    with connect_database() as connection:
        rows = connection.execute(
            "SELECT id, title, body FROM notes ORDER BY id"
        ).fetchall()
    return [dict(row) for row in rows]


def get_note(note_id: int) -> dict[str, object] | None:
    with connect_database() as connection:
        row = connection.execute(
            "SELECT id, title, body FROM notes WHERE id = ?",
            (note_id,),
        ).fetchone()
    return dict(row) if row is not None else None


note_id = create_note("Repository pattern", "SQL belongs outside the route.")
print("Created note:", get_note(note_id))
print("All notes:", list_notes())


## 4. Expose the repository through FastAPI

FastAPI maps Python functions to HTTP routes. Pydantic validates the incoming body. The status code documents the outcome.

We use `TestClient` below: it exercises the ASGI application directly, so you do not need to launch Uvicorn for this lab.


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI(title="Class 2 notes lab")


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


@app.post("/notes", status_code=201)
def create_note_route(note: NoteCreate) -> dict[str, object]:
    new_id = create_note(note.title, note.body)
    created = get_note(new_id)
    assert created is not None
    return created


@app.get("/notes")
def list_notes_route() -> dict[str, list[dict[str, object]]]:
    return {"items": list_notes()}


@app.get("/notes/{note_id}")
def get_note_route(note_id: int) -> dict[str, object]:
    note = get_note(note_id)
    if note is None:
        raise HTTPException(status_code=404, detail="Note not found")
    return note


client = TestClient(app)
print(client.get("/health").status_code, client.get("/health").json())


### Exercise: inspect the HTTP contract

Run the next cell. Pay attention to the difference between the success, validation and not-found responses.


In [ ]:
created_response = client.post(
    "/notes",
    json={"title": "FastAPI", "body": "Routes map Python functions to HTTP."},
)
print("Create:", created_response.status_code, created_response.json())

print("List:", client.get("/notes").status_code, client.get("/notes").json())

invalid_response = client.post(
    "/notes",
    json={"title": "", "body": "The title is deliberately empty."},
)
print("Invalid payload:", invalid_response.status_code, invalid_response.json())

missing_response = client.get("/notes/9999")
print("Missing note:", missing_response.status_code, missing_response.json())

assert created_response.status_code == 201
assert invalid_response.status_code == 422
assert missing_response.status_code == 404


## 5. Keyword search and safe SQL

A simple keyword search can use SQLite `LIKE`. This is not semantic search: it finds literal substrings, not concepts or synonyms.

The query remains a parameter. Try different phrases and observe which notes are returned.


In [ ]:
def search_notes(query: str) -> list[dict[str, object]]:
    if not query.strip():
        raise ValueError("query must not be empty")
    with connect_database() as connection:
        rows = connection.execute(
            """
            SELECT id, title, body
            FROM notes
            WHERE title LIKE ? OR body LIKE ?
            ORDER BY id
            """,
            (f"%{query}%", f"%{query}%"),
        ).fetchall()
    return [dict(row) for row in rows]


print(search_notes("HTTP"))
print(search_notes("missing"))


## 6. Test behavior, not implementation details

A useful API test checks the contract visible to a client: method, path, status code and response body. It should not depend on private local variable names inside a route.


In [ ]:
def test_notes_api_contract() -> None:
    response = client.post(
        "/notes",
        json={"title": "API test", "body": "The contract is observable."},
    )
    assert response.status_code == 201
    payload = response.json()
    assert payload["title"] == "API test"
    assert isinstance(payload["id"], int)

    assert client.get(f"/notes/{payload['id']}").status_code == 200
    assert client.get("/notes/99999").status_code == 404
    assert client.post("/notes", json={"title": "", "body": "bad"}).status_code == 422


test_notes_api_contract()
print("API contract checks passed.")


## 7. Try it: add a DELETE endpoint

Use the existing repository and route patterns to add `delete_note(note_id)` and `DELETE /notes/{note_id}`.

Suggested behavior:

- Return HTTP `204` when a note is deleted.
- Return HTTP `404` when the ID does not exist.
- Add a test for both cases.
- Do not put SQL directly inside the route.

This is a short in-class extension. It is not a solution to the Christmas project.


In [ ]:
# TODO — implement these two pieces:
#
# 1. delete_note(note_id: int) -> bool
#    Use a parameterized DELETE statement and return whether a row was removed.
#
# 2. DELETE /notes/{note_id}
#    Call delete_note(), return an empty 204 response on success,
#    and raise HTTPException(404, ...) if no row was removed.
#
# Add a TestClient test for success and not-found behavior.


## 8. Run an API server and inspect its documentation

For an actual local server, define the app in a Python module and run:

```bash
uv run uvicorn your_module:app --reload
```

Then open:

- `http://127.0.0.1:8000/health`
- `http://127.0.0.1:8000/docs` — interactive Swagger UI
- `http://127.0.0.1:8000/openapi.json` — generated OpenAPI document

In the Engineering Knowledge Assistant homework, the API will expose documents and search results instead of notes.


## Wrap-up

Before leaving, check that you can explain:

1. The difference between an HTTP method, path, status code and JSON body.
2. Why a Pydantic model is useful at an API boundary.
3. Why SQL belongs in a repository/data-access layer rather than route handlers.
4. Why SQL values must be parameterized.
5. What a primary key, foreign key and unique constraint protect.
6. Why `TestClient` can test an API without launching Uvicorn.
7. Why literal keyword search is not the same as semantic search.